
#  Personal Chef AI Agent

An AI-powered personal chef that turns the ingredients you already have into recipe recommendations.

This project uses an AI agent to analyze an image of available ingredients, identify the items in the image, search the web for relevant recipes, and recommend a recipe that makes the best use of the available ingredients.

**Main goal:** Help users decide what to cook based on the ingredients they already have, minimizing the need to purchase additional items.


In [ ]:

## How It Works

The agent follows a three-step workflow:

1. **Ingredient Recognition:** Analyzes the provided image and identifies the available ingredients.
2. **Recipe Search:** Uses a web search tool to find recipes containing the identified ingredients.
3. **Recipe Selection:** Selects and presents a recipe that uses the largest possible number of available ingredients.

The agent uses the image as its source of available ingredients and prioritizes recipes that require minimal additional items.



## Instructions and Constraints

The agent follows these guidelines when recommending recipes:

- It does not need to use every ingredient identified in the image.
- It should prioritize recipes that include as many available ingredients as possible.
- It should avoid suggesting ingredients that are not present in the image.
- Additional ingredients should be kept to a minimum.
- The final recommendation should include the recipe name, ingredients, preparation instructions, and source when available.


In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [ ]:
from langchain.agents import create_agent

system_prompt = """
You are a chef who will receive an image containing the available ingredients and must:
Identify the ingredients present in the image.
Search the web for recipes that include the identified ingredients.
Return a recipe that uses the largest possible number of ingredients identified in the image.
Rules:
You do not need to use all the ingredients; these are simply the ingredients available.
Avoid using ingredients that are not shown in the image. The goal is to prepare a recipe using the available ingredients.
"""

agent = create_agent(
    model='gpt-5-nano',
    tools=[web_search],
    system_prompt=system_prompt
)

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

In [ ]:
print(uploader.value)

In [ ]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
multimodal_question = HumanMessage(content=[
    # {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

In [ ]:
import sqlite3

conn = sqlite3.connect("receitas.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS receitas (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    titulo TEXT,
    receita TEXT
)
""")

conn.commit()
receita = response["messages"][-1].content
titulo = receita.split("\n")[0]
cursor.execute(
    """
    INSERT INTO receitas (titulo, receita)
    VALUES (?, ?)
    """,
    (titulo, receita)
)

conn.commit()


cursor.execute(
    "SELECT receita FROM receitas WHERE id = 1"
)

print(cursor.fetchone()[0])